In [1]:
import numpy as np
import pandas as pd
import neurokit2 as nk
from pathlib import Path
from tqdm import tqdm
import time

from scipy.signal import resample_poly

WINDOWING SAMPLERATE AND OVERLAP

In [21]:
FS = 200                     # Sampling frequency (Hz)
WINDOW_SEC = 60             # Sliding window length (seconds)
WINDOW_SAMPLES = FS * WINDOW_SEC
STEP_SEC = 15                # Step size (seconds)


 Resample signal using polyphase filtering

In [3]:
def resample_signal(signal, fs_in=256, fs_out=200):

    return resample_poly(signal, fs_out, fs_in)

In [4]:
import warnings
warnings.filterwarnings("ignore")


FEATURE EXTRACTION VER 1

In [5]:
def extract_features_from_full_gsr_ppg(df, participant_id):

    # -------------------------
    # Convert signals
    # -------------------------
    ppg = pd.to_numeric(df["ppg"], errors="coerce")
    eda = pd.to_numeric(df["gsr"], errors="coerce")

    ppg = ppg.interpolate().bfill().ffill()
    eda = eda.interpolate().bfill().ffill()

    # Resample signals to 200 Hz

    ppg = resample_signal(ppg.values, 256, 200)
    eda = resample_signal(eda.values, 256, 200)

    # -------------------------
    # Process signals
    # -------------------------
    signals_ppg, info_ppg = nk.ppg_process(ppg, sampling_rate=FS)
    signals_eda, _ = nk.eda_process(eda, sampling_rate=FS)

    # -------------------------
    # RR intervals (IBI )
    # -------------------------
    rpeaks = info_ppg["PPG_Peaks"]
    rr_intervals = np.diff(rpeaks) / FS * 1000
    rr_times = rpeaks[1:]

    features = []
    step_samples = FS * STEP_SEC

    for start in range(0, len(df) - WINDOW_SAMPLES, step_samples):

        end = start + WINDOW_SAMPLES
        window = {}

        # HR
        hr_window = signals_ppg["PPG_Rate"].iloc[start:end]
        window["HR"] = hr_window.mean(skipna=True)

        # HRV
        rr_mask = (rr_times >= start) & (rr_times < end)
        rr_win = rr_intervals[rr_mask]

        if len(rr_win) >= 5:
            diff_rr = np.diff(rr_win)
            window["HRV_RMSSD"] = np.sqrt(np.mean(diff_rr ** 2))
            window["HRV_SDNN"] = np.std(rr_win, ddof=1)
        else:
            window["HRV_RMSSD"] = np.nan
            window["HRV_SDNN"] = np.nan

        # EDA
        eda_window = signals_eda.iloc[start:end]

        window["EDA_Tonic"] = eda_window["EDA_Tonic"].mean()
        window["EDA_Phasic"] = eda_window["EDA_Phasic"].mean()
        window["SCR_Count"] = int(eda_window["SCR_Peaks"].sum())

        # Metadata
        window["participant"] = participant_id
        window["time_sec"] = start // FS

        features.append(window)

    return pd.DataFrame(features)

FEATURE EXTRACTION WITH RR VALUES FOR ANALYSIS and SCR_RATE

In [8]:
def extract_features_with_rr(df, participant_id):

    # -------------------------
    # Convert signals
    # -------------------------
    ppg = pd.to_numeric(df["ppg"], errors="coerce")
    eda = pd.to_numeric(df["gsr"], errors="coerce")

    ppg = ppg.interpolate().bfill().ffill()
    eda = eda.interpolate().bfill().ffill()

    # -------------------------
    # Resample signals to 200 Hz
    # -------------------------
    ppg = resample_signal(ppg.values, 256, 200)
    eda = resample_signal(eda.values, 256, 200)

    # -------------------------
    # Process signals
    # -------------------------
    signals_ppg, info_ppg = nk.ppg_process(ppg, sampling_rate=FS)
    signals_eda, _ = nk.eda_process(eda, sampling_rate=FS)

    # -------------------------
    # RR intervals (IBI)
    # -------------------------
    rpeaks = info_ppg["PPG_Peaks"]

    rr_intervals = np.diff(rpeaks) / FS * 1000 # ms

    # bättre tidsrepresentation (mitten av intervallet)
    rr_times = (rpeaks[1:] + rpeaks[:-1]) / 2

    features = []

    step_samples = FS * STEP_SEC
    signal_len = len(ppg)  # 🔥 FIX: använd resamplad signal

    # -------------------------
    # Sliding windows
    # -------------------------
    for start in range(0, signal_len - WINDOW_SAMPLES, step_samples):

        end = start + WINDOW_SAMPLES
        window = {}

        # -------------------------
        # HR
        # -------------------------
        hr_window = signals_ppg["PPG_Rate"].iloc[start:end]
        window["HR"] = hr_window.mean(skipna=True)

        # -------------------------
        # HRV + RR diagnostics
        # -------------------------
        rr_mask = (rr_times >= start) & (rr_times < end)
        rr_win = rr_intervals[rr_mask]

        # filtrera orimliga RR
        rr_win = rr_win[(rr_win > 300) & (rr_win < 1500)]

        if len(rr_win) >= 5:
            diff_rr = np.diff(rr_win)

            # filtrera extrema diff
            diff_rr = diff_rr[np.abs(diff_rr) < 200]

            # HRV
            window["HRV_RMSSD"] = np.sqrt(np.mean(diff_rr ** 2))
            window["HRV_SDNN"] = np.std(rr_win, ddof=1)

            # -------------------------
            # RR diagnostics 
            # -------------------------
            window["RR_mean"] = np.mean(rr_win)
            window["RR_std"] = np.std(rr_win)
            window["RR_min"] = np.min(rr_win)
            window["RR_max"] = np.max(rr_win)
            window["RR_count"] = len(rr_win)
            window["RR_diff_std"] = np.std(diff_rr)

            window["RR_valid"] = 1

        else:
            window["HRV_RMSSD"] = np.nan
            window["HRV_SDNN"] = np.nan

            window["RR_mean"] = np.nan
            window["RR_std"] = np.nan
            window["RR_min"] = np.nan
            window["RR_max"] = np.nan
            window["RR_count"] = 0
            window["RR_diff_std"] = np.nan

            window["RR_valid"] = 0

        # -------------------------
        # EDA
        # -------------------------
        eda_window = signals_eda.iloc[start:end]

        window["EDA_Tonic"] = eda_window["EDA_Tonic"].mean()
        window["EDA_Phasic"] = eda_window["EDA_Phasic"].mean()
       
        scr_count = int(eda_window["SCR_Peaks"].sum())

        window["SCR_Rate"] = scr_count/ WINDOW_SEC

        

        # -------------------------
        # Metadata
        # -------------------------
        window["participant"] = participant_id
        window["time_sec"] = start // FS

        features.append(window)

    return pd.DataFrame(features)

In [12]:
def extract_features_with_PPG_foucus(df, participant_id):

    # -------------------------
    # Convert signals
    # -------------------------
    ppg = pd.to_numeric(df["ppg"], errors="coerce")
    eda = pd.to_numeric(df["gsr"], errors="coerce")

    ppg = ppg.interpolate().bfill().ffill()
    eda = eda.interpolate().bfill().ffill()

    # -------------------------
    # Resample signals to 200 Hz
    # -------------------------
    ppg = resample_signal(ppg.values, 256, 200)
    eda = resample_signal(eda.values, 256, 200)

    # -------------------------
    # Remove edge artifacts
    # -------------------------
    ppg = ppg[FS*2:-FS*2]
    eda = eda[FS*2:-FS*2]

    # -------------------------
    # Process signals (bättre peak detection)
    # -------------------------
    signals_ppg, info_ppg = nk.ppg_process(
        ppg,
        sampling_rate=FS,
        method="elgendi",   
        method_peaks="charlton"   # bättre än default
    )

    signals_eda, _ = nk.eda_process(eda, sampling_rate=FS)

    signal_clean = signals_ppg["PPG_Clean"].values
    rpeaks = info_ppg["PPG_Peaks"]

    # -------------------------
    # Peak refinement (KRITISK)
    # -------------------------
    def refine_peaks(signal, peaks, fs, window=0.05):
        refined = []
        w = int(window * fs)

        for p in peaks:
            start = max(0, p - w)
            end = min(len(signal), p + w)

            segment = signal[start:end]
            new_p = start + np.argmax(segment)

            refined.append(new_p)

        return np.array(refined)

    rpeaks = refine_peaks(signal_clean, rpeaks, FS)

    # -------------------------
    # RR intervals (IBI)
    # -------------------------
    rr_intervals = np.diff(rpeaks) / FS * 1000  # ms

    # bättre tidsrepresentation (mitten av intervallet)
    rr_times = (rpeaks[1:] + rpeaks[:-1]) / 2

    features = []

    step_samples = FS * STEP_SEC
    signal_len = len(signal_clean)  # viktigt

    # -------------------------
    # Sliding windows
    # -------------------------
    for start in range(0, signal_len - WINDOW_SAMPLES, step_samples):

        end = start + WINDOW_SAMPLES
        window = {}

        # -------------------------
        # HR
        # -------------------------
        hr_window = signals_ppg["PPG_Rate"].iloc[start:end]
        window["HR"] = hr_window.mean(skipna=True)

        # -------------------------
        # HRV + RR diagnostics
        # -------------------------
        rr_mask = (rr_times >= start) & (rr_times < end)
        rr_win = rr_intervals[rr_mask]

        # -------------------------
        # Strikt RR filtering
        # -------------------------
        rr_win = rr_win[(rr_win > 400) & (rr_win < 1200)]

        if len(rr_win) >= 10:

            diff_rr = np.diff(rr_win)

            # bättre diff-filter
            valid_diff = np.abs(diff_rr) < 150
            diff_rr = diff_rr[valid_diff]

            # quality metric
            rr_quality = np.sum(valid_diff) / len(valid_diff)

            # -------------------------
            # HRV
            # -------------------------
            if len(diff_rr) > 0:
                window["HRV_RMSSD"] = np.sqrt(np.mean(diff_rr ** 2))
                window["RR_diff_std"] = np.std(diff_rr)
            else:
                window["HRV_RMSSD"] = np.nan
                window["RR_diff_std"] = np.nan

            window["HRV_SDNN"] = np.std(rr_win, ddof=1)

            # -------------------------
            # RR diagnostics 
            # -------------------------
            window["RR_mean"] = np.mean(rr_win)
            window["RR_std"] = np.std(rr_win)
            window["RR_min"] = np.min(rr_win)
            window["RR_max"] = np.max(rr_win)
            window["RR_count"] = len(rr_win)
            window["RR_quality"] = rr_quality
            window["RR_valid"] = 1

        else:
            window["HRV_RMSSD"] = np.nan
            window["HRV_SDNN"] = np.nan

            window["RR_mean"] = np.nan
            window["RR_std"] = np.nan
            window["RR_min"] = np.nan
            window["RR_max"] = np.nan
            window["RR_count"] = 0
            window["RR_diff_std"] = np.nan
            window["RR_quality"] = 0
            window["RR_valid"] = 0

        # -------------------------
        # EDA 
        # -------------------------
        eda_window = signals_eda.iloc[start:end]

        window["EDA_Tonic"] = eda_window["EDA_Tonic"].mean()
        window["EDA_Phasic"] = eda_window["EDA_Phasic"].mean()

        scr_count = int(eda_window["SCR_Peaks"].sum())
        window["SCR_Rate"] = scr_count / WINDOW_SEC

        # -------------------------
        # Metadata
        # -------------------------
        window["participant"] = participant_id
        window["time_sec"] = start // FS

        features.append(window)

    return pd.DataFrame(features)

FEATURE EXTRACTION WITH FOCUS ON PPG VALUES.

USING CENTROID TO CALCULATE RR-->HRV

In [52]:
def extract_features_with_PPG_foucus_centroid(df, participant_id):

    
    # Convert signals to NaN if not numeric    
    ppg = pd.to_numeric(df["ppg"], errors="coerce")
    eda = pd.to_numeric(df["gsr"], errors="coerce")


    # fill missing values
    ppg = ppg.interpolate().bfill().ffill()
    eda = eda.interpolate().bfill().ffill()

   
    # Resample signals to 200 Hz with scipy's resample_poly. 
    ppg = resample_signal(ppg.values, 256, 200)
    eda = resample_signal(eda.values, 256, 200)

    
    # To remove unstable edges start/end (2sec). 
    ppg = ppg[FS*2:-FS*2]
    eda = eda[FS*2:-FS*2]

    print("\n=== SIGNAL INFO ===")
    print("Signal length (samples):", len(ppg))
    print("Signal length (seconds):", len(ppg) / FS)

    
    # Process signals using NeuroKit2's PPG processing.    
    signals_ppg, info_ppg = nk.ppg_process(
        ppg,
        sampling_rate=FS,
        method="elgendi",
        method_peaks="charlton"
    )

    signals_eda, _ = nk.eda_process(eda, sampling_rate=FS)

    signal_clean = signals_ppg["PPG_Clean"].values
    rpeaks = info_ppg["PPG_Peaks"]

  
    #  Centroid refinement
    # -------------------------
    def refine_peaks_centroid(signal, peaks, fs, window=0.05):
        refined = []
        w = int(window * fs)

        for p in peaks:
            start = max(0, p - w)
            end = min(len(signal), p + w)

            segment = signal[start:end]
            x = np.arange(start, end)

            # gör signal positiv
            y = segment - np.min(segment)

            # fallback om något går fel
            if np.sum(y) == 0 or len(segment) == 0:
                refined.append(p)
                continue

            centroid = np.sum(x * y) / np.sum(y)

            refined.append(int(centroid))

        return np.array(refined)

    # använd centroid istället för max
    rpeaks = refine_peaks_centroid(signal_clean, rpeaks, FS)

    # -------------------------
    # RR intervals (IBI)
    # -------------------------
    rr_intervals = np.diff(rpeaks) / FS * 1000  # ms

    # bättre tidsrepresentation
    rr_times = (rpeaks[1:] + rpeaks[:-1]) / 2

    features = []

    step_samples = FS * STEP_SEC
    signal_len = len(signal_clean)

    print("\n=== WINDOW SETTINGS ===")
    print("Window samples:", WINDOW_SAMPLES)
    print("Step samples:", step_samples)
    print("Step (seconds):", step_samples / FS)
    print("Expected windows (rough):", int((signal_len - WINDOW_SAMPLES) / step_samples + 1))

    # -------------------------
    # Sliding windows
    # -------------------------
    for start in range(0, signal_len - WINDOW_SAMPLES + 1, step_samples):

        end = start + WINDOW_SAMPLES
        window = {}

        # -------------------------
        # HR
        # -------------------------
        hr_window = signals_ppg["PPG_Rate"].iloc[start:end]
        window["HR"] = hr_window.mean(skipna=True)

        # -------------------------
        # RR selection
        # -------------------------
        rr_mask = (rr_times >= start) & (rr_times < end)
        rr_win = rr_intervals[rr_mask]

        # -------------------------
        # RR filtering
        # -------------------------
        rr_win = rr_win[(rr_win > 400) & (rr_win < 1200)]

        if len(rr_win) >= 10:

            diff_rr = np.diff(rr_win)

            valid_diff = np.abs(diff_rr) < 150
            diff_rr = diff_rr[valid_diff]

            rr_quality = np.sum(valid_diff) / len(valid_diff)

            # -------------------------
            # HRV
            # -------------------------
            if len(diff_rr) > 0:
                window["HRV_RMSSD"] = np.sqrt(np.mean(diff_rr ** 2))
                window["RR_diff_std"] = np.std(diff_rr)
            else:
                window["HRV_RMSSD"] = np.nan
                window["RR_diff_std"] = np.nan

            window["HRV_SDNN"] = np.std(rr_win, ddof=1)

            # -------------------------
            # RR diagnostics
            # -------------------------
            window["RR_mean"] = np.mean(rr_win)
            window["RR_std"] = np.std(rr_win)
            window["RR_min"] = np.min(rr_win)
            window["RR_max"] = np.max(rr_win)
            window["RR_count"] = len(rr_win)
            window["RR_quality"] = rr_quality
            window["RR_valid"] = 1

        else:
            window["HRV_RMSSD"] = np.nan
            window["HRV_SDNN"] = np.nan
            window["RR_mean"] = np.nan
            window["RR_std"] = np.nan
            window["RR_min"] = np.nan
            window["RR_max"] = np.nan
            window["RR_count"] = 0
            window["RR_diff_std"] = np.nan
            window["RR_quality"] = 0
            window["RR_valid"] = 0

        # -------------------------
        # EDA
        # -------------------------
        eda_window = signals_eda.iloc[start:end]

        window["EDA_Tonic"] = eda_window["EDA_Tonic"].mean()
        window["EDA_Phasic"] = eda_window["EDA_Phasic"].mean()

        scr_count = int(eda_window["SCR_Peaks"].sum())
        window["SCR_Rate"] = scr_count / WINDOW_SEC

        # -------------------------
        # Metadata
        # -------------------------
        window["participant"] = participant_id
        window["time_sec"] = start // FS

        features.append(window)

    return pd.DataFrame(features)

FEATURE EXTRACTION FOR TRAIN PART 1-48 FULL FILE (full_gsr_ppg.csv)

In [57]:
DATA_DIR = Path("../data/raw/Participants")

# -------------------------
# Collect participant files
# -------------------------
participant_files = sorted(
    DATA_DIR.rglob("full_gsr_ppg*.csv"),
    key=lambda x: int(x.parent.name.replace("Part", ""))
)
print("Found files:", len(participant_files))
for f in participant_files[:5]:
    print(f)
# -------------------------
# Select participants 1–48
# -------------------------
participant_files = participant_files[:48]

all_features = []

start_time = time.time()

for i, file in enumerate(tqdm(participant_files, desc="Processing participants")):
    participant_id = file.parent.name

    print(f"\nProcessing {participant_id} ({i+1}/{len(participant_files)})")

    df = pd.read_csv(file)

    participant_features = extract_features_with_PPG_foucus_centroid(
        df, participant_id
    )

    all_features.append(participant_features)

    # -------------------------
    # Time estimation
    # -------------------------
    elapsed = time.time() - start_time
    avg_time = elapsed / (i + 1)
    remaining = avg_time * (len(participant_files) - (i + 1))

    print(
        f"Elapsed: {elapsed/60:.1f} min | "
        f"Remaining: {remaining/60:.1f} min"
    )


Found files: 60
..\data\raw\Participants\Part1\full_gsr_ppg.csv
..\data\raw\Participants\Part2\full_gsr_ppg.csv
..\data\raw\Participants\Part3\full_gsr_ppg.csv
..\data\raw\Participants\Part4\full_gsr_ppg.csv
..\data\raw\Participants\Part5\full_gsr_ppg.csv


Processing participants:   0%|          | 0/48 [00:00<?, ?it/s]


Processing Part1 (1/48)

=== SIGNAL INFO ===
Signal length (samples): 434867
Signal length (seconds): 2174.335

=== WINDOW SETTINGS ===
Window samples: 12000
Step samples: 3000
Step (seconds): 15.0
Expected windows (rough): 141


Processing participants:   2%|▏         | 1/48 [00:59<46:18, 59.12s/it]

Elapsed: 1.0 min | Remaining: 46.3 min

Processing Part2 (2/48)

=== SIGNAL INFO ===
Signal length (samples): 467179
Signal length (seconds): 2335.895

=== WINDOW SETTINGS ===
Window samples: 12000
Step samples: 3000
Step (seconds): 15.0
Expected windows (rough): 152


...

Elapsed: 39.2 min | Remaining: 1.7 min

Processing Part47 (47/48)

=== SIGNAL INFO ===
Signal length (samples): 465123
Signal length (seconds): 2325.615

=== WINDOW SETTINGS ===
Window samples: 12000
Step samples: 3000
Step (seconds): 15.0
Expected windows (rough): 152


Processing participants:  98%|█████████▊| 47/48 [39:50<00:42, 42.30s/it]

Elapsed: 39.8 min | Remaining: 0.8 min

Processing Part48 (48/48)

=== SIGNAL INFO ===
Signal length (samples): 465093
Signal length (seconds): 2325.465

=== WINDOW SETTINGS ===
Window samples: 12000
Step samples: 3000
Step (seconds): 15.0
Expected windows (rough): 152


Processing participants: 100%|██████████| 48/48 [41:20<00:00, 51.68s/it]

Elapsed: 41.3 min | Remaining: 0.0 min


FEATURE EXTRACTION WITH SPECIAL ORDER IN CSV FILE

In [55]:
from pathlib import Path
import pandas as pd
import time
from tqdm import tqdm

# 🔴 NY DATAKÄLLA (din processed data)
DATA_DIR = Path("../data/processed/fix_order")

# =========================================
# VÄLJ PARTICIPANTS HÄR
# =========================================

# Alternativ 1: EN deltagare
selected_ids = [49]

# Alternativ 2: ALLA
# selected_ids = list(range(49, 61))

# Alternativ 3: SPANN
# selected_ids = list(range(5, 21))  # 5–20

# Alternativ 4: specifika
# selected_ids = [1, 3, 7, 12]


# =========================================
# 🔴 HÄMTA FILER
# =========================================

participant_files = []

for pid in selected_ids:
    file_path = DATA_DIR / f"participant_{pid}.csv"
    
    if file_path.exists():
        participant_files.append((pid, file_path))
    else:
        print(f"⚠️ Saknar fil för participant {pid}")

print("Antal filer:", len(participant_files))

# =========================================
# 🔴 PROCESSING
# =========================================

all_features = []
start_time = time.time()

for i, (participant_id, file) in enumerate(
    tqdm(participant_files, desc="Processing participants")
):
    print(f"\nProcessing participant_{participant_id} ({i+1}/{len(participant_files)})")

    df = pd.read_csv(file)

    # 🔴 DIN FEATURE FUNCTION
    participant_features = extract_features_with_PPG_foucus_centroid(
        df, participant_id
    )

    all_features.append(participant_features)

    # -------------------------
    # ⏱️ Time estimation
    # -------------------------
    elapsed = time.time() - start_time
    avg_time = elapsed / (i + 1)
    remaining = avg_time * (len(participant_files) - (i + 1))

    print(
        f"Elapsed: {elapsed/60:.1f} min | "
        f"Remaining: {remaining/60:.1f} min"
    )

Antal filer: 1


Processing participants:   0%|          | 0/1 [00:00<?, ?it/s]


Processing participant_49 (1/1)

=== SIGNAL INFO ===
Signal length (samples): 204470
Signal length (seconds): 1022.35


Processing participants:   0%|          | 0/1 [00:03<?, ?it/s]


KeyboardInterrupt: 

DF CHECK BEFORE LOG AND CLIPPING 

In [60]:
features_df = pd.concat(all_features, ignore_index=True)
features_df = features_df.dropna().reset_index(drop=True)
features_df.describe()


,HR,HRV_RMSSD,RR_diff_std,HRV_SDNN,RR_mean,RR_std,RR_min,RR_max,RR_count,RR_quality,RR_valid,EDA_Tonic,EDA_Phasic,SCR_Rate,time_sec
count,7140.000000,7140.000000,7140.000000,7140.000000,7140.000000,7140.000000,7140.000000,7140.000000,7140.000000,7140.000000,7140.0,7140.000000,7140.000000,7140.000000,7140.000000
mean,79.532862,48.666082,48.498450,62.790035,772.820892,62.374425,616.009804,942.745098,78.707423,0.945109,1.0,2628.261978,-0.074314,0.052787,1111.815126
std,11.474293,16.384805,16.233063,23.965499,105.082721,23.790148,105.865234,129.856098,10.947671,0.082891,0.0,11561.380223,40.390525,0.052657,649.214749
min,54.150973,8.767506,8.722435,15.359078,449.492188,15.287805,405.000000,510.000000,48.000000,0.450704,1.0,-242.369799,-1831.157036,0.000000,0.000000
25%,71.556682,35.784869,35.754667,45.615142,699.636628,45.337381,540.000000,855.000000,71.000000,0.923077,1.0,84.949633,-0.060222,0.000000,555.000000
50%,78.297697,49.229266,49.107919,59.214789,768.365385,58.819692,625.000000,945.000000,78.000000,0.984127,1.0,172.277890,0.000291,0.033333,1110.000000
75%,86.337564,61.513781,61.198903,76.881464,839.014085,76.310617,690.000000,1035.000000,85.000000,1.000000,1.0,369.844768,0.064420,0.083333,1665.000000
max,141.942360,94.688192,94.607741,183.023187,1093.750000,181.465514,990.000000,1195.000000,128.000000,1.000000,1.0,91350.056346,1817.296738,0.300000,2265.000000


LOG AND CLIPPING FEATURES 

In [61]:

# EDA stabilization, log-transform and clip extreme values

EPS = 1e-6

features_df["EDA_Tonic_log"] = np.log(features_df["EDA_Tonic"] - features_df["EDA_Tonic"].min() + EPS)
features_df["EDA_Phasic_log"] = np.log(np.abs(features_df["EDA_Phasic"]) + EPS)

# SCR LOG
features_df["SCR_Rate"] = np.log1p(features_df["SCR_Rate"])

#Clip extreme values

features_df["HRV_RMSSD"] = features_df["HRV_RMSSD"].clip(0,200)
features_df["HRV_SDNN"] = features_df["HRV_SDNN"].clip(0,200)


# EDA CLIPP
features_df["EDA_Tonic_log"]  = features_df["EDA_Tonic_log"].clip(6.0, 8.0)
features_df["EDA_Phasic_log"] = features_df["EDA_Phasic_log"].clip(-4.0, 2.0)

# SCR CLIPP

features_df["SCR_Rate"] = features_df["SCR_Rate"].clip(0, 0.2)


SAVE FULL FILE FOR ANALYS

In [62]:
features_df.to_csv("../data/features_1_48_60ws_15ol.csv", index=False)

CREATE DF FOR SOM MODEL WITH CORRECT FEATURES

In [ ]:
# Feature columns used for SOM
FEATURE_COLS = [
    "HR",
    "HRV_RMSSD",
    "HRV_SDNN",
    "SCR_Rate",
    "EDA_Tonic_log",
    "EDA_Phasic_log"
]

# SOM input matrix
X_features = features_df[FEATURE_COLS]

# (Optional) metadata kept separately
metadata = features_df[["participant", "time_sec"]]

X_features.describe()

In [ ]:
print(features_df["SCR_Rate"].describe())

In [ ]:
features_df = features_df[features_df["RR_quality"] > 0.7]

# Feature columns used for SOM
FEATURE_COLS = [
    "HR",
    "HRV_RMSSD",
    
]

# SOM input
X_features_PPG = features_df[FEATURE_COLS]

# metadata
metadata = features_df[["participant", "time_sec"]]

# check
X_features_PPG.describe()

SAVE FULL FILE FOR SOM MODEL

In [37]:
X_features_PPG.to_csv(    
    "../data/test_features_SOM_fix_order.csv",
    index=False
)


FEATURE EXTRACTION 49-60 FOR TEST with clipping and log

FULL FILE (full_gsr_ppg.csv)

SAVED IN /test_features_windows --> 1: all participants 2: participant_49..60

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import time
from tqdm import tqdm
import os

# -------------------------
# SETTINGS
# -------------------------
DATA_DIR = Path("../data/raw/Participants")
SAVE_DIR = Path("features_win_60")
os.makedirs(SAVE_DIR, exist_ok=True)

EPS = 1e-6

# -------------------------
# Collect participant files
# -------------------------
participant_files = sorted(
    DATA_DIR.rglob("full_gsr_ppg*.csv"),
    key=lambda x: int(x.parent.name.replace("Part", ""))
)

print("Total files found:", len(participant_files))

# -------------------------
# Select participants 49–60
# -------------------------
participant_files = participant_files[48:60]

print("Selected participants:")
for f in participant_files:
    print(f.parent.name)

# -------------------------
# PROCESSING
# -------------------------
all_features = []
start_time = time.time()

for i, file in enumerate(tqdm(participant_files, desc="Processing participants")):

    participant_id = int(file.parent.name.replace("Part", ""))
    tqdm.write(f"Processing Part{participant_id} ({i+1}/{len(participant_files)})")

    df = pd.read_csv(file)

    # -------------------------
    # FEATURE EXTRACTION WITH RR
    # -------------------------
    features_df = extract_features_with_PPG_foucus_centroid(df, participant_id)
    features_df = features_df[
    (features_df["RR_quality"] > 0.7)
    ]

    # -------------------------
    # EDA STABILIZATION 
    # -------------------------
    features_df["EDA_Tonic_log"] = np.log(
        features_df["EDA_Tonic"] - features_df["EDA_Tonic"].min() + EPS
    )

    features_df["EDA_Phasic_log"] = np.log(
        np.abs(features_df["EDA_Phasic"]) + EPS
    )

    # -------------------------
    # CLIPPING 
    # -------------------------
    features_df["HRV_RMSSD"] = features_df["HRV_RMSSD"].clip(0, 200)
    features_df["HRV_SDNN"]  = features_df["HRV_SDNN"].clip(0, 200)

   
    # SCR LOG
    features_df["SCR_Rate"] = np.log1p(features_df["SCR_Rate"])
    features_df["SCR_Rate"] = features_df["SCR_Rate"].clip(0, 0.2)

    

    # EDA clipping 
    features_df["EDA_Tonic_log"]  = features_df["EDA_Tonic_log"].clip(6.0, 8.0)
    features_df["EDA_Phasic_log"] = features_df["EDA_Phasic_log"].clip(-4.0, 2.0)

    # -------------------------
    # DROP RAW EDA 
    # -------------------------
    features_df = features_df.drop(columns=["EDA_Tonic", "EDA_Phasic"])

    # -------------------------
    # SAVE PER PARTICIPANT
    # -------------------------
    features_df.to_csv(
        SAVE_DIR / f"participant_{participant_id}_windows_features_PPG_centroid.csv",
        index=False
    )

    all_features.append(features_df)

    # -------------------------
    # Time estimation
    # -------------------------
    elapsed = time.time() - start_time
    avg_time = elapsed / (i + 1)
    remaining = avg_time * (len(participant_files) - (i + 1))

    tqdm.write(
        f"Elapsed: {elapsed/60:.1f} min | Remaining: {remaining/60:.1f} min"
    )

# -------------------------
# COMBINE ALL
# -------------------------
features_all_df = pd.concat(all_features, ignore_index=True)

features_all_df.to_csv(
    SAVE_DIR / "features_win_60_sec_all.csv",
    index=False
)

print("\n✅ DONE")
print(features_all_df.shape)

FEATURE EXTRACTION BY_BLOCK 

FILE PATH -->  file = p_dir / "by_block" / "7_gsr_ppg_.csv" (CHANGE FOR DIFFERENT BLOCK)

SAVE BOTH ALL PARTICIPANT AND ONE PER PARTICIPANT 

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import time
from tqdm import tqdm
import os

# -------------------------
# SETTINGS
# -------------------------
DATA_DIR = Path("../data/raw/Participants")
SAVE_DIR = Path("test_features_neutral_7_btw_stroop_and_iq")
os.makedirs(SAVE_DIR, exist_ok=True)

EPS = 1e-6

# -------------------------
# Collect stroop files
# -------------------------
participant_dirs = sorted(
    [p for p in DATA_DIR.glob("Part*")],
    key=lambda x: int(x.name.replace("Part", ""))
)

# Select 49–60
participant_dirs = participant_dirs[48:60]

print("Selected participants:")
for p in participant_dirs:
    print(p.name)

# -------------------------
# PROCESSING
# -------------------------
all_features = []
start_time = time.time()

for i, p_dir in enumerate(tqdm(participant_dirs, desc="Processing participants")):

    participant_id = int(p_dir.name.replace("Part", ""))
    tqdm.write(f"Processing Part{participant_id} ({i+1}/{len(participant_dirs)})")

    # -------------------------
    # FILE PATH (stroop)
    # -------------------------
    file = p_dir / "by_block" / "7_gsr_ppg_.csv"

    if not file.exists():
        tqdm.write(f"⚠️ Missing file for Part{participant_id}")
        continue

    df = pd.read_csv(file)

    # -------------------------
    # FEATURE EXTRACTION
    # -------------------------
    features_df = extract_features_with_PPG_foucus_centroid(df, participant_id)

    print("Shape:", features_df.shape)
    print("Columns:", features_df.columns)

    # -------------------------
    # EDA STABILIZATION
    # -------------------------
    features_df["EDA_Tonic_log"] = np.log(
        features_df["EDA_Tonic"] - features_df["EDA_Tonic"].min() + EPS
    )

    features_df["EDA_Phasic_log"] = np.log(
        np.abs(features_df["EDA_Phasic"]) + EPS
    )

    # -------------------------
    # CLIPPING
    # -------------------------
    features_df["HRV_RMSSD"] = features_df["HRV_RMSSD"].clip(0, 200)
    features_df["HRV_SDNN"]  = features_df["HRV_SDNN"].clip(0, 200)

    # SCR LOG
    features_df["SCR_Rate"] = np.log1p(features_df["SCR_Rate"])
    features_df["SCR_Rate"] = features_df["SCR_Rate"].clip(0, 0.2)

    features_df["EDA_Tonic_log"]  = features_df["EDA_Tonic_log"].clip(6.0, 8.0)
    features_df["EDA_Phasic_log"] = features_df["EDA_Phasic_log"].clip(-4.0, 2.0)

    # -------------------------
    # DROP RAW EDA
    # -------------------------
    features_df = features_df.drop(columns=["EDA_Tonic", "EDA_Phasic"])

    # -------------------------
    # SAVE PER PARTICIPANT
    # -------------------------
    features_df.to_csv(
        SAVE_DIR / f"participant_{participant_id}_neutral_7_btw_stroop_and_iq.csv",
        index=False
    )

    all_features.append(features_df)

    # -------------------------
    # Time estimation
    # -------------------------
    elapsed = time.time() - start_time
    avg_time = elapsed / (i + 1)
    remaining = avg_time * (len(participant_dirs) - (i + 1))

    tqdm.write(
        f"Elapsed: {elapsed/60:.1f} min | Remaining: {remaining/60:.1f} min"
    )

# -------------------------
# COMBINE ALL
# -------------------------
features_all_df = pd.concat(all_features, ignore_index=True)

features_all_df.to_csv(
    SAVE_DIR / "Neutral_7_BTW_Stroop_And_IQ_ALL_WINDOWS.csv",
    index=False
)

print("\n✅ DONE")
print(features_all_df.shape)